# Chapter 1. 머신러닝 문제 정식화와 세 갈래 분류 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter01_2_taxonomy.ipynb)

책 본문: [1.2 머신러닝 문제 정식화와 세 갈래 분류](https://smhanlab.com/book-ml/kor/ml1/chapter01.html)

이 노트북은 책 1.2절의 핵심 아이디어를 장난감 데이터로 직접 만듭니다:
(1) 세 갈래 학습의 "데이터 형상"을 numpy로 만들고, (2) "라벨이 있는가? 보상이
있는가?" 두 질문만으로 데이터를 분류하는 함수를 짜 보고, (3) 같은 z가 회귀로도
분류로도 읽히는 일(시그모이드 링크함수)을 코드로 확인하고, (4) "같은 점군을
라벨이 있는/없는 관점에서 보는" 차이를 시각화합니다.

## 1. 세 갈래 장난감 데이터셋 만들기

책에서 강조했듯, 세 갈래를 가르는 기준은 알고리즘의 이름이 아니라
**데이터의 형상**입니다. 각 갈래의 데이터가 실제로 어떻게 "보이는가"를
가장 작은 규모로 만들어봅니다(시드 고정으로 언제 실행해도 동일).

In [ ]:
import numpy as np

rng = np.random.default_rng(42)  # 고정 시드: 언제 실행하든 같은 데이터

# (1) 지도학습 · 회귀: 평수(m²) -> 가격(백만 원)
X_sup_reg = rng.uniform(50, 150, 40)
y_sup_reg = 3.0 * X_sup_reg + 50 + rng.normal(0, 20, 40)

# (2) 지도학습 · 분류: 두 특징 -> 'a'/'b' 범주
X_sup_cls = rng.normal([0, 0], 1.0, (30, 2))
y_sup_cls = np.array(["a", "b"] * 15)

# (3) 비지도학습: 특징만, 라벨 열이 아예 없음
X_unsup = rng.normal([0, 0], 1.0, (30, 2))

# (4) 강화학습: (상태, 행동, 보상) 경험 튜플의 목록
rl_traces = [(0, "right", -1.0), (1, "right", -1.0), (2, "stop", 10.0)]

print("지도·회귀 샘플:  x =", round(X_sup_reg[0], 2), "  y =", round(y_sup_reg[0], 1))
print("지도·분류 샘플:  x =", X_sup_cls[0].round(2), "  y =", y_sup_cls[0])
print("비지도  샘플:    x =", X_unsup[0].round(2), "  (y 열이 아예 없음)")
print("강화학습 샘플:   (상태, 행동, 보상) =", rl_traces[0])

## 2. 두 질문으로 갈래 판정

책의 "세 갈래를 한 줄로 구분하는 두 가지 질문"을 그대로 함수로 옮깁니다:
**Q1. y(라벨)가 있는가? → 있으면 지도학습(라벨이 숫자인가 범주인가?).**
**Q2. 없으면, (상태, 행동, 보상) 튜플인가? → 있으면 강화학습, 없으면 비지도.**

In [ ]:
def classify_dataset(data):
    """세 갈래를 가르는 두 질문: 1) y(라벨)가 있는가? 2) (상태,행동,보상)인가?"""
    if isinstance(data, dict) and "y" in data:
        is_numeric = np.issubdtype(np.asarray(data["y"]).dtype, np.number)
        return "지도학습 · 회귀" if is_numeric else "지도학습 · 분류"
    if isinstance(data, list) and data and isinstance(data[0], tuple) and len(data[0]) == 3:
        return "강화학습 (상태, 행동, 보상)"
    return "비지도학습 (x만)"

datasets = {
    "평수 -> 가격":    {"X": X_sup_reg, "y": y_sup_reg},
    "특징 -> a/b":     {"X": X_sup_cls, "y": y_sup_cls},
    "라벨 없는 특징":  X_unsup,
    "경험 튜플 목록":  rl_traces,
}
expected = {
    "평수 -> 가격":   "지도학습 · 회귀",
    "특징 -> a/b":    "지도학습 · 분류",
    "라벨 없는 특징": "비지도학습 (x만)",
    "경험 튜플 목록": "강화학습 (상태, 행동, 보상)",
}
for name, ds in datasets.items():
    verdict = classify_dataset(ds)
    print(f"{name:<10} -> {verdict}")
    assert verdict == expected[name], f"{name}: {verdict}"
print("\n4개 데이터셋 모두 두 질문만으로 올바르게 분류됨")

## 3. 회귀 vs 분류: 같은 z, 두 가지 해석

책의 손 계산을 그대로 코드로 옮겨봅니다: \(w=[0.5,-2.0], b=1.0,
x=[3,0.5]\)일 때 \(z = w^Tx + b = 1.5\). 이 하나의 \(z\)를 **그대로**
읽으면 회귀 예측값이고, **시그모이드에 통과시키면** 분류 확률이 됩니다.

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 그래프의 한글 라벨이 깨지지 않도록 CJK 폰트 사용 (Colab에 기본 설치됨)
for _f in ["Noto Sans CJK KR", "NanumGothic", "Malgun Gothic", "AppleGothic"]:
    if any(_f.lower() == x.name.lower() for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

w = [0.5, -2.0]
b = 1.0
x = [3, 0.5]
z = sum(wi * xi for wi, xi in zip(w, x)) + b
print(f"z = w·x + b = {z}")
print(f"회귀로 읽으면:  {z}        (예: '이 집은 1.5억')")
print(f"분류로 읽으면:  P(스팸) = sigmoid({z}) = {sigmoid(z):.4f}  (≈82%)")
assert abs(z - 1.5) < 1e-12
assert abs(sigmoid(z) - 0.8175745) < 1e-6

zs = np.linspace(-4, 4, 200)
plt.figure(figsize=(6, 4))
plt.plot(zs, [sigmoid(zz) for zz in zs], "b-")
plt.plot([1.5], [sigmoid(1.5)], "ro", markersize=8)
plt.annotate("z = 1.5 → 0.818", (1.5, sigmoid(1.5)),
             xytext=(0.2, 0.95), arrowprops=dict(arrowstyle="->"))
plt.xlabel("z = w·x + b")
plt.ylabel("σ(z)")
plt.title("같은 z: 그대로 쓰면 회귀, σ()에 통과시키면 분류")
plt.grid(alpha=0.3)
plt.show()

## 4. 같은 점군, 라벨이 있는/없는 관점

책의 "같은 데이터, 다른 갈래" 예시입니다. 같은 점군을 (왼쪽) 라벨을
붙인 분류 관점과 (오른쪽) 라벨을 버린 비지도 관점으로 나란히 보고,
라벨 없이도 점군 자체가 두 그룹으로 모여 있음(= 구조)을 확인합니다.
아래의 초간단 k-2-means(초기 중심 고정, 5회 반복)는 라벨 없이도
같은 두 그룹의 중심으로 수렴함을 보여줍니다 — Chapter 14에서 이
알고리즘을 체계적으로 다룹니다.

In [ ]:
# 라벨이 있는 점군: 두 가우시안 클러스터
A = rng.normal([1, 1], 0.4, (15, 2))
B = rng.normal([4, 4], 0.4, (15, 2))
X_pts = np.vstack([A, B])
y_pts = np.array([0] * 15 + [1] * 15)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(X_pts[:, 0], X_pts[:, 1], c=y_pts, cmap="viridis", s=40)
axes[0].set_title("Supervised view: y (label) attached")
axes[1].scatter(X_pts[:, 0], X_pts[:, 1], c="gray", s=40)
axes[1].set_title("Unsupervised view: same points, no label")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 라벨 없이 k=2로 클러스터 중심을 손으로 찾아보기 (초기 중심 고정)
centers = np.array([[1.0, 1.0], [4.0, 4.0]])
for _ in range(5):
    dists = np.linalg.norm(X_pts[:, None, :] - centers[None, :, :], axis=2)
    labels = dists.argmin(axis=1)
    for k in range(2):
        centers[k] = X_pts[labels == k].mean(axis=0)

print("라벨 없이 찾은 중심 (k-2-means):")
for k, c in enumerate(centers):
    print(f"  클러스터 {k}: {c.round(3)}")
print("라벨이 있을 때의 실제 중심:")
for k in range(2):
    print(f"  클러스터 {k}: {X_pts[y_pts == k].mean(axis=0).round(3)}")
for k in range(2):
    true_c = X_pts[y_pts == k].mean(axis=0)
    nearest = min(centers, key=lambda c: np.linalg.norm(c - true_c))
    assert np.linalg.norm(nearest - true_c) < 0.1
print("\n라벨 없이 찾은 구조가, 라벨이 있을 때와 같은 두 그룹으로 수렴")